# 🎨 AUTO-SCRIBE V2: TRUNG TÂM ĐIỀU KHIỂN & CẦU NỐI AI AGENT LAPTOP
Hệ thống tự động hóa Whiteboard Animation với **Giao diện Web tương tác (Chạy 1 Cell)**:
- 🤖 **Agent-in-the-Loop (0 API Key)**: Colab gửi câu thoại về Laptop ➔ Bạn nhờ Antigravity (AI trong IDE) phân tích sâu sắc kịch bản ➔ Colab kéo kết quả về đóng gói VideoScribe!
- ⚡ **Drive In-Memory Cache**: Quét kho ảnh Drive trong 0.5s, không lo nghẽn mạng.
- 🎨 **Tự Động Sinh Ảnh SVG Doodle (Pollinations + vtracer)**: Tự động vẽ và lưu vào Drive `f/gen/`.
- 📦 **Xuất File VideoScribe (.scribe) 1-Click**.

## ⚡ BƯỚC 1: Cài Đặt Môi Trường & Kết Nối Google Drive (Chạy 1 lần)

In [ ]:
from google.colab import drive
import os
import sys

print("🔗 Đang yêu cầu quyền truy cập Google Drive...")
drive.mount('/content/drive')

print("⏳ Đang cài đặt thư viện lõi (Whisper, Gemini, vtracer, Gradio, Pillow, ffmpeg)...")
!apt-get install -y ffmpeg
!pip install -q openai-whisper google-genai requests vtracer Pillow gradio

print("✅ Đã cài đặt xong toàn bộ môi trường! Hãy chuyển sang BƯỚC 2 để mở Giao Diện.")

## 🎛️ BƯỚC 2: Khởi Chạy Giao Diện Web Điều Khiển Toàn Diện (All-In-One UI)

In [ ]:
import os
import re
import json
import time
import random
import shutil
import zipfile
import subprocess
import urllib.request
import urllib.parse
from PIL import Image
import vtracer
import whisper
import requests
from google import genai
from google.genai import types
import gradio as gr

ASSETS_DIR = "assets"
os.makedirs(ASSETS_DIR, exist_ok=True)

def clean_slug(text):
    text = re.sub(r'[^a-zA-Z0-9\s_-]', '', text)
    text = re.sub(r'\s+', '_', text).strip('_').lower()
    return text[:40] if text else "doodle_icon"

def videoscribe_escape(s):
    s = s.replace('&', '&amp;')
    s = s.replace('<', '&lt;')
    s = s.replace('"', '&quot;')
    return s

def build_drive_cache(drive_search_dirs):
    cache = []
    for s_dir in drive_search_dirs:
        if os.path.exists(s_dir):
            for root, dirs, files in os.walk(s_dir):
                for f in files:
                    if f.lower().endswith('.svg') or f.lower().endswith('.png'):
                        clean_n = re.sub(r'[^a-zA-Z0-9]', ' ', os.path.splitext(f)[0]).lower()
                        cache.append({
                            "path": os.path.join(root, f),
                            "filename": f,
                            "words": set(clean_n.split()),
                            "clean_name": clean_n
                        })
    return cache

def generate_doodle_svg(prompt_keyword, target_svg_path, drive_save_path=None):
    os.makedirs(os.path.dirname(os.path.abspath(target_svg_path)), exist_ok=True)
    ai_prompt = (
        f"clean black and white whiteboard doodle line art, minimalist sketch drawing of {prompt_keyword}, "
        f"continuous clean black ink outline on pure white background, minimal vector icon, no color, no shading, high contrast, clean strokes"
    )
    encoded_prompt = urllib.parse.quote(ai_prompt)
    seed = random.randint(1000, 999999)
    url = f"https://image.pollinations.ai/prompt/{encoded_prompt}?width=768&height=768&model=flux&nologo=true&seed={seed}"
    
    temp_img = target_svg_path + ".temp_dl"
    temp_png = target_svg_path + ".temp.png"
    success = False
    
    for attempt in range(3):
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=25) as resp:
                with open(temp_img, 'wb') as f:
                    f.write(resp.read())
            
            with Image.open(temp_img) as img:
                img = img.convert('RGB')
                img.save(temp_png, 'PNG')
            
            vtracer.convert_image_to_svg_py(
                temp_png,
                target_svg_path,
                colormode='binary',
                hierarchical='stacked',
                mode='spline',
                filter_speckle=4,
                corner_threshold=60,
                length_threshold=4.0,
                max_iterations=10,
                splice_threshold=45,
                path_precision=3
            )
            success = True
            break
        except Exception:
            time.sleep(1.5)
        finally:
            if os.path.exists(temp_img): os.remove(temp_img)
            if os.path.exists(temp_png): os.remove(temp_png)
            
    if success and os.path.exists(target_svg_path) and os.path.getsize(target_svg_path) > 100:
        if drive_save_path:
            try:
                os.makedirs(os.path.dirname(os.path.abspath(drive_save_path)), exist_ok=True)
                shutil.copy(target_svg_path, drive_save_path)
            except Exception: pass
        return target_svg_path
    else:
        with open(target_svg_path, "w", encoding="utf-8") as f:
            f.write(f'<svg xmlns="http://www.w3.org/2000/svg" width="500" height="500"><rect width="500" height="500" fill="none" stroke="#000" stroke-width="4"/><text x="250" y="250" font-size="26" text-anchor="middle" fill="#000">{prompt_keyword}</text></svg>')
        return target_svg_path

def search_or_generate_svg_fast(query, drive_cache, drive_gen_dir, target_asset_path, used_files=None):
    if used_files is None: used_files = set()
    stop_words = {"vector", "illustration", "clipart", "transparent", "icon", "svg", "drawing", "the", "a", "an"}
    raw_words = re.sub(r'[^a-zA-Z0-9]', ' ', query).lower().split()
    query_words = set([w for w in raw_words if w not in stop_words and len(w) > 1])
    
    best_matches = []
    max_score = 0
    
    for item in drive_cache:
        score = len(query_words.intersection(item["words"]))
        if score > 0:
            if " ".join(query_words) in item["clean_name"]: score += 2.0
            if score > max_score:
                max_score = score
                best_matches = [item["path"]]
            elif score == max_score:
                best_matches.append(item["path"])
                
    if best_matches and max_score >= 1.0:
        unused = [m for m in best_matches if m not in used_files]
        chosen = random.choice(unused) if unused else random.choice(best_matches)
        used_files.add(chosen)
        shutil.copy(chosen, target_asset_path)
        return target_asset_path, f"Drive: {os.path.basename(chosen)}"
    
    slug_n = clean_slug(query)
    drive_save = os.path.join(drive_gen_dir, f"{slug_n}.svg") if drive_gen_dir else None
    if drive_save and os.path.exists(drive_save):
        drive_save = os.path.join(drive_gen_dir, f"{slug_n}_{random.randint(100,999)}.svg")
        
    generate_doodle_svg(query, target_asset_path, drive_save)
    time.sleep(0.5)
    return target_asset_path, f"AI Mới: {os.path.basename(drive_save) if drive_save else 'Local'}"

def clean_json_response(text):
    text = text.strip()
    if text.startswith("```json"): text = text[7:]
    elif text.startswith("```"): text = text[3:]
    if text.endswith("```"): text = text[:-3]
    return text.strip()

def build_scribe_file(audio_path, metadata_path="scene_metadata.json"):
    if not os.path.exists(metadata_path): return None
    with open(metadata_path, "r", encoding="utf-8") as f:
        meta_data = json.load(f)

    BUILD_DIR = "build_scribe"
    if os.path.exists(BUILD_DIR): shutil.rmtree(BUILD_DIR)
    os.makedirs(BUILD_DIR, exist_ok=True)

    audio_dst = os.path.join(BUILD_DIR, "voiceover.mp3")
    if audio_path and os.path.exists(audio_path):
        try:
            subprocess.run(['ffmpeg', '-y', '-i', audio_path, '-ar', '44100', '-ac', '2', '-b:a', '192k', audio_dst], stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)
        except Exception:
            shutil.copy(audio_path, audio_dst)

    drawing_xml = os.path.join(BUILD_DIR, "drawing.xml")
    with open(drawing_xml, "w", encoding="utf-8") as f:
        f.write('<?xml version="1.0" encoding="utf-8"?>\n')
        f.write('<drawing visualScale="1.0" canvasType="0" canvasColor="-1" bgFitMode="stretch" version="3.7.3103" resolution="1080" voiceoverVolume="100" soundtrackVolume="100">\n')
        
        xml_elements = []
        element_counter = 1000000000 + random.randint(10000, 99999)
        visual_timeline_ms = 0
        
        for scene_idx, scene in enumerate(meta_data):
            raw_images = scene.get('images', [])
            n = len(raw_images)
            if n == 0: continue
            
            speech_dur_s = scene['end'] - scene['start']
            duration_per_img = speech_dur_s / n
            scene_x = scene_idx * 1600
            scene_y = 0
            cam_scale = 0.82
            cam_x = 448.5 - scene_x * cam_scale
            cam_y = 252.5 - scene_y * cam_scale
            
            for i, img_meta in enumerate(raw_images):
                filename = img_meta.get('file_name', '')
                file_path = os.path.join(ASSETS_DIR, filename)
                alt_svg = os.path.splitext(file_path)[0] + ".svg"
                if os.path.exists(alt_svg): file_path = alt_svg
                elif not os.path.exists(file_path): continue
                
                is_svg = file_path.endswith('.svg')
                actual_filename = os.path.basename(file_path)
                
                if not is_svg:
                    vec_svg = file_path + ".vectorized.svg"
                    try:
                        vtracer.convert_image_to_svg_py(file_path, vec_svg, colormode='binary', hierarchical='stacked', mode='spline')
                        file_path = vec_svg
                    except Exception: pass

                with open(file_path, "r", encoding="utf-8") as f2:
                    raw_svg = f2.read()
                    raw_svg = re.sub(r'<\?xml[^>]*\?>', '', raw_svg)
                    raw_svg = re.sub(r'<!DOCTYPE[^>]*>', '', raw_svg)
                    raw_svg = re.sub(r'<!--.*?-->', '', raw_svg, flags=re.DOTALL)
                    content = raw_svg.replace('\n', ' ').replace('\r', '')
                    
                element_counter += random.randint(1000, 5000)
                if n == 1: pos_x, pos_y, scale_val = scene_x, scene_y, "0.8"
                elif n == 2: pos_x, pos_y, scale_val = scene_x + (-250 if i == 0 else 250), scene_y, "0.55"
                else: pos_x, pos_y, scale_val = scene_x + (-220 if i == 1 else (220 if i == 2 else 0)), scene_y + (120 if i > 0 else -120), "0.45"
                    
                ai_style = img_meta.get('animation_style', 'draw')
                if ai_style == 'draw': draw_style = 'draw_style_normal'
                elif ai_style in ['movein', 'movein_hand', 'movein_nohand']: draw_style = 'draw_style_movein'
                elif ai_style == 'fadein': draw_style = 'draw_style_fadein'
                else: draw_style = 'draw_style_normal'
                
                movin_compass = str(random.randint(1, 8))
                draw_detail = 'yes' if draw_style == 'draw_style_normal' else 'no'
                custom_hand = 'default_nohand' if draw_style == 'draw_style_movein' else ''
                movin_arc = random.choice(['0', '1']) if draw_style == 'draw_style_movein' else '0'
                
                total_time_ms = int(duration_per_img * 1000)
                trans_time_ms = min(500, int(total_time_ms * 0.15))
                pause_time_ms = min(500, int(total_time_ms * 0.10))
                target_time_ms = max(0, total_time_ms - trans_time_ms - pause_time_ms)
                
                drawing_xml_attr = f'drawingXML="{videoscribe_escape(content)}"' if is_svg else f'drawingXML="{videoscribe_escape(content)}" imageRef="{actual_filename}"'
                
                element_xml = (
                    f'  <element elementType="drawing" descName="" elementID="{element_counter}" '
                    f'splitTextField="no" drawingText="" fontName="null" {drawing_xml_attr} '
                    f'customHandMD5="{custom_hand}" colourEffect="0" targetTime="{target_time_ms}" '
                    f'pauseTime="{pause_time_ms}" transitionTime="{trans_time_ms}" '
                    f'drawStyle="{draw_style}" rotation="0" visible="true" '
                    f'currentPosX="{pos_x}" currentPosY="{pos_y}" offsetX="{pos_x}" offsetY="{pos_y}" '
                    f'scalesX="0.8" scalesY="0.8" theScale="0.8" targetHeight="800" '
                    f'movinCompass="{movin_compass}" movinFlow="0" movinArc="{movin_arc}" movinAllowRotate="yes" '
                    f'drawDetail="{draw_detail}" sketchStyle="no" brush="0" brushOptions="0" opacity="1" '
                    f'textColour="-1" textAlign="left" textBackwards="no" rtlLanguage="no" textSpacing="0" '
                    f'flipHoriz="no" flipVert="no" locked="no" calligraphy_angle="45" keepRunning="no" '
                    f'loopOptions="Fit to Time" blendMode="normal" filters="&lt;filters/>" morphFromID="0" '
                    f'morphCamera="no" morphRemoveOld="yes" cameraPositionX="{cam_x}" cameraPositionY="{cam_y}" '
                    f'cameraScale="{cam_scale}" cameraCanvasWid="897.7777777777778" cameraCanvasHei="505" '
                    f'availableRecolours="&lt;availableRecolours/>" recolouringSchemes="&lt;recolouringSchemes/>" '
                    f'skinTone="-1" hairColour="-1" highlightColour="-1" customColour1="-1" customColour2="-1" '
                    f'originalOutlineColour="0" greyscaleContrast="70" />\n'
                )
                xml_elements.append(element_xml)
                visual_timeline_ms += total_time_ms
                
        f.write('\n'.join(xml_elements) + '\n')
        f.write('</drawing>')

    out_file = "Auto_Project.scribe"
    with zipfile.ZipFile(out_file, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(BUILD_DIR):
            for file in files:
                f_p = os.path.join(root, file)
                zipf.write(f_p, os.path.relpath(f_p, BUILD_DIR))

    tmp_p = out_file + ".tmp"
    with zipfile.ZipFile(out_file, 'r') as zin, zipfile.ZipFile(tmp_p, 'w', zipfile.ZIP_DEFLATED) as zout:
        for item in zin.infolist():
            data = zin.read(item.filename)
            if item.filename == 'drawing.xml':
                data = data.decode('utf-8').replace('&gt;', '>').encode('utf-8')
            zout.writestr(item, data)
    os.replace(tmp_p, out_file)
    return out_file

def generate_preview_table(meta_data=None):
    if meta_data is None:
        if not os.path.exists("scene_metadata.json"): return "<p>Chưa có dữ liệu kịch bản.</p>"
        with open("scene_metadata.json", "r", encoding="utf-8") as f:
            meta_data = json.load(f)
            
    rows = ""
    for s in meta_data:
        imgs_div = ""
        for img in s.get('images', []):
            fp = os.path.join(ASSETS_DIR, img.get('file_name', ''))
            svg_content = ""
            if os.path.exists(fp) and fp.endswith('.svg'):
                try:
                    with open(fp, "r", encoding="utf-8") as svg_f:
                        svg_content = svg_f.read()
                        svg_content = re.sub(r'<\?xml[^>]*\?>', '', svg_content)
                except Exception: pass
                    
            imgs_div += f'''
            <div style="background:#fff; border:1px solid #e2e8f0; border-radius:6px; padding:6px; margin:4px; display:inline-block; vertical-align:top; width:110px; text-align:center;">
                <div style="height:70px; display:flex; align-items:center; justify-content:center; overflow:hidden;">{svg_content if svg_content else '<div style="color:#a0aec0;">IMG</div>'}</div>
                <div style="font-size:11px; font-weight:bold; color:#2d3748; overflow:hidden; text-overflow:ellipsis; white-space:nowrap;">{img.get('svg_search_prompt','')}</div>
                <div style="font-size:10px; color:#4a5568;">{img.get('animation_style','draw')}</div>
                <div style="font-size:9px; color:#3182ce;">{img.get('source','')}</div>
            </div>
            '''
        rows += f'''
        <tr style="border-bottom:1px solid #edf2f7;">
            <td style="padding:10px; font-weight:bold; color:#4a5568; vertical-align:top; width:50px;">#{s['sentence_id']}</td>
            <td style="padding:10px; vertical-align:top; width:80px; font-size:12px; color:#718096;">{s['start']:.1f}s ➔ {s['end']:.1f}s</td>
            <td style="padding:10px; vertical-align:top; color:#2d3748; font-size:13px; line-height:1.4;">{s['speech_text']}</td>
            <td style="padding:10px; vertical-align:top;">{imgs_div}</td>
        </tr>
        '''
    return f'''
    <div style="font-family:-apple-system, sans-serif; background:#fff; border-radius:8px; border:1px solid #e2e8f0; max-height:450px; overflow-y:auto;">
        <table style="width:100%; border-collapse:collapse; text-align:left;">
            <thead>
                <tr style="background:#f7fafc; color:#718096; font-size:11px; text-transform:uppercase; border-bottom:2px solid #e2e8f0;">
                    <th style="padding:10px;">ID</th><th style="padding:10px;">Thời Gian</th><th style="padding:10px;">Câu Thoại</th><th style="padding:10px;">Ảnh SVG & Hiệu Ứng</th>
                </tr>
            </thead>
            <tbody>{rows}</tbody>
        </table>
    </div>
    '''

# --- BƯỚC 1: BÓC TÁCH WHISPER & GỬI VỀ LAPTOP ---
def step1_whisper_and_send_to_laptop(audio_file_obj, bridge_url_input):
    logs = []
    def fmt_log(msg):
        logs.append(f"[{time.strftime('%H:%M:%S')}] {msg}")
        return "\n".join(logs)

    audio_path = audio_file_obj if isinstance(audio_file_obj, str) else (audio_file_obj.name if audio_file_obj else None)
    if not audio_path and os.path.exists("voiceover.mp3"): audio_path = "voiceover.mp3"
    if not audio_path or not os.path.exists(audio_path):
        yield fmt_log("❌ Lỗi: Vui lòng chọn file âm thanh voiceover.mp3!")
        return

    bridge_url = bridge_url_input.strip().rstrip('/') if bridge_url_input else ""
    if not bridge_url:
        yield fmt_log("❌ Lỗi: Vui lòng dán Local Bridge URL (Cloudflare Tunnel) từ Laptop!")
        return

    yield fmt_log(f"🎙️ Âm thanh: {os.path.basename(audio_path)}")
    yield fmt_log("⏳ Whisper đang bóc tách câu thoại...")
    whisper_model = whisper.load_model("base")
    whisper_res = whisper_model.transcribe(audio_path)
    scenes = [{"sentence_id": idx+1, "start": seg["start"], "end": seg["end"], "text": seg["text"].strip()} for idx, seg in enumerate(whisper_res["segments"]) if seg["text"].strip()]
    yield fmt_log(f"✅ Đã bóc tách {len(scenes)} câu thoại!")

    yield fmt_log(f"🌐 Đang gửi {len(scenes)} câu thoại về Laptop qua {bridge_url}...")
    try:
        resp = requests.post(f"{bridge_url}/", json={"scenes": scenes}, timeout=30)
        if resp.status_code == 200:
            yield fmt_log("═"*60)
            yield fmt_log(f"🎉 ĐÃ GỬI THÀNH CÔNG VỀ LAPTOP (File: pending_scenes.json)!")
            yield fmt_log("👉 BÂY GIỜ BẠN HÃY BẢO AI TRONG IDE LAPTOP: 'Hãy phân tích kịch bản pending_scenes.json'.")
            yield fmt_log("👉 Sau khi AI trong IDE phân tích xong, bạn chỉ việc bấm nút 'BƯỚC 2: Nhận Kịch Bản & Xuất Video' bên dưới!")
            yield fmt_log("═"*60)
        else:
            yield fmt_log(f"⚠️ Laptop trả về mã lỗi: {resp.status_code}")
    except Exception as e:
        yield fmt_log(f"❌ Lỗi gửi về Laptop: {e}")

# --- BƯỚC 2: KÉO KỊCH BẢN TỪ LAPTOP VỀ & XUẤT VIDEOSCRIBE ---
def step2_pull_from_laptop_and_build(audio_file_obj, bridge_url_input, drive_f_path, drive_gen_path):
    logs = []
    def fmt_log(msg):
        logs.append(f"[{time.strftime('%H:%M:%S')}] {msg}")
        return "\n".join(logs)

    bridge_url = bridge_url_input.strip().rstrip('/') if bridge_url_input else ""
    if not bridge_url:
        yield fmt_log("❌ Lỗi: Vui lòng dán Local Bridge URL!"), None, None
        return

    yield fmt_log(f"📥 Đang kéo kịch bản đã phân tích từ Laptop ({bridge_url}/get_analyzed_scenes)..."), None, None
    try:
        resp = requests.get(f"{bridge_url}/get_analyzed_scenes", timeout=30)
        if resp.status_code != 200 or resp.json().get("status") != "success":
            yield fmt_log("⚠️ Chưa có kịch bản trên Laptop! Hãy bảo AI trong IDE: 'Hãy phân tích kịch bản pending_scenes.json' trước!"), None, None
            return
        raw_analyzed_data = resp.json().get("data", [])
        yield fmt_log(f"🎉 Đã nhận thành công kịch bản của {len(raw_analyzed_data)} cảnh từ Laptop!"), None, None
    except Exception as e:
        yield fmt_log(f"❌ Lỗi kết nối tới Laptop: {e}"), None, None
        return

    # Quét cache Drive
    yield fmt_log("📂 Đang quét kho ảnh trên Google Drive..."), None, None
    drive_search_dirs = [drive_f_path, drive_gen_path]
    os.makedirs(drive_gen_path, exist_ok=True)
    drive_cache = build_drive_cache(drive_search_dirs)
    yield fmt_log(f"✅ Tìm thấy {len(drive_cache)} ảnh có sẵn trong Drive!"), None, None

    # Tải / Sinh SVG
    scene_metadata = []
    used_files = set()
    total_scenes = len(raw_analyzed_data)

    for j, s in enumerate(raw_analyzed_data):
        sc_id = s.get("sentence_id", j + 1)
        raw_imgs = s.get("images", [])
        sentence_entry = {"sentence_id": sc_id, "start": s.get('start', 0), "end": s.get('end', 0), "speech_text": s.get('speech_text', s.get('text', '')), "images": []}
        
        for img_idx, img_info in enumerate(raw_imgs):
            file_base = f"sentence_{sc_id:03d}_img_{img_idx+1:02d}"
            kw = img_info.get("svg_search_prompt", "icon")
            target_p = os.path.join(ASSETS_DIR, f"{file_base}.svg")
            
            act_p, src_note = search_or_generate_svg_fast(kw, drive_cache, drive_gen_path, target_p, used_files)
            
            sentence_entry["images"].append({
                "img_idx": img_idx + 1,
                "visual_concept": img_info.get("visual_concept", ""),
                "svg_search_prompt": kw,
                "animation_style": img_info.get("animation_style", "draw"),
                "file_name": os.path.basename(act_p),
                "source": src_note
            })
            
        scene_metadata.append(sentence_entry)
        yield fmt_log(f"   [Cảnh {sc_id}/{total_scenes}] ➔ {len(raw_imgs)} ảnh ({src_note})"), None, None
        
    with open("scene_metadata.json", "w", encoding="utf-8") as f:
        json.dump(scene_metadata, f, ensure_ascii=False, indent=2)

    # Đóng gói
    audio_path = audio_file_obj if isinstance(audio_file_obj, str) else (audio_file_obj.name if audio_file_obj else "voiceover.mp3")
    yield fmt_log("📦 Đang đóng gói dự án Auto_Project.scribe..."), None, None
    out_scribe = build_scribe_file(audio_path, "scene_metadata.json")
    
    yield fmt_log("✨ HOÀN TẤT 100%! Bạn có thể tải file .scribe ở khung bên phải."), out_scribe, generate_preview_table(scene_metadata)

# --- GIAO DIỆN WEB GRADIO ---
with gr.Blocks(title="Auto-Scribe V2 Control Center", theme=gr.themes.Soft(primary_hue="blue", neutral_hue="slate")) as app:
    gr.Markdown("# 🚀 AUTO-SCRIBE V2: TRUNG TÂM ĐIỀU KHIỂN & CẦU NỐI AI AGENT")
    gr.Markdown("Hệ thống tự động hóa Whiteboard: Colab gửi câu thoại về Laptop ➔ Bạn nhờ AI trong IDE phân tích ➔ Colab kéo về xuất VideoScribe (0đ API Key).")
    
    with gr.Tabs():
        with gr.TabItem("🤖 Quy Trình Với AI Agent Laptop (0đ API Key)"):
            with gr.Row():
                with gr.Column(scale=1):
                    audio_in = gr.File(label="🎙️ File Giọng Đọc (voiceover.mp3)", file_types=[".mp3", ".wav"])
                    bridge_url_in = gr.Textbox(label="🔗 Local Bridge URL (Cloudflare Tunnel từ Laptop)", placeholder="https://xxxx.trycloudflare.com")
                    drive_f_in = gr.Textbox(label="📂 Thư mục ảnh gốc trên Drive", value="/content/drive/MyDrive/image/f")
                    drive_gen_in = gr.Textbox(label="💾 Thư mục lưu ảnh AI sinh mới trên Drive", value="/content/drive/MyDrive/image/f/gen")
                    
                    gr.Markdown("### 📌 BƯỚC 1: Bóc tách giọng nói & Gửi về Laptop")
                    btn_step1 = gr.Button("📤 1. Whisper Bóc Tách & Gửi Kịch Bản Về Laptop", variant="primary")
                    
                    gr.Markdown("### 📌 BƯỚC 2: Nhận kết quả từ Laptop & Đóng gói Video")
                    btn_step2 = gr.Button("📥 2. Kéo Kịch Bản Đã Phân Tích & Xuất VideoScribe", variant="secondary")
                
                with gr.Column(scale=1):
                    log_box = gr.Textbox(label="📋 Nhật Ký Hoạt Động (Live Console Logs)", lines=16, interactive=False)
                    out_file = gr.File(label="📦 Tải File Dự Án VideoScribe (.scribe)")
                    
        with gr.TabItem("🖼️ Xem Trước & Sửa Ảnh Trực Quan (Visual Editor)"):
            preview_display = gr.HTML(label="Bảng Kịch Bản")
            
    btn_step1.click(
        fn=step1_whisper_and_send_to_laptop,
        inputs=[audio_in, bridge_url_in],
        outputs=[log_box]
    )
    
    btn_step2.click(
        fn=step2_pull_from_laptop_and_build,
        inputs=[audio_in, bridge_url_in, drive_f_in, drive_gen_in],
        outputs=[log_box, out_file, preview_display]
    )

print("🌐 Đang khởi chạy Giao Diện Web & Mở Đường Link Tunnel...")
app.queue().launch(share=True, debug=False, show_error=True)